In [1]:
import networkx as nx
import msgpack

    # Importing Matplotlib, Pandas, and NumPy for logs parsing and visualization
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import sys
import os

In [2]:
root_dir = os.path.dirname(os.path.dirname(os.path.abspath("./test1_data.json")))
sys.path.append(root_dir)

from edge_sim_py import *
from edge_sim_py import flow_scheduling

In [3]:
#Dispaly components
def Collect_Components()->dict:
    datasets={}
    #User
    datasets[f"{User.__name__}"]=[]
    for user in User.all():
        datasets[f"{user.__class__.__name__}"].append((user._to_dict()))
        
    datasets[f"{Application.__name__}"]=[]
    for app in Application.all():
        datasets[f"{app.__class__.__name__}"].append((app._to_dict()))
        
    datasets[f"{Service.__name__}"]=[]
    for service in Service.all():
        datasets[f"{service.__class__.__name__}"].append((service._to_dict()))
        
    datasets[f"{EdgeServer.__name__}"]=[]
    for server in EdgeServer.all():
        datasets[f"{server.__class__.__name__}"].append((server._to_dict()))
       
    datasets[f"{BaseStation.__name__}"]=[]    
    for station in BaseStation.all():
        datasets[f"{station.__class__.__name__}"].append((station._to_dict()))  
        
    datasets[f"{NetworkSwitch.__name__}"]=[]         
    for switch in NetworkSwitch.all():
        datasets[f"{switch.__class__.__name__}"].append((switch._to_dict()))
        
    datasets[f"{NetworkLink.__name__}"]=[]           
    for link in NetworkLink.all():
        datasets[f"{link.__class__.__name__}"].append((link._to_dict()))  

    return datasets
#

def printComponent(datasets,category:str):
    print(f"{category}:")
    for agent in datasets[category]:
        print(agent)
        print()
    
def printClass(category):
    print(category.__name__)
    #User
    for agent in category.all():
        print(agent._to_dict())
        print()
        
    

In [4]:
#2.步进测试
# 先替换各类的步进函数
#User
User.step = tools.User_step
User.set_communication_path = tools.User_path
User._compute_delay = tools.User_compute_delay
#app and service
Application.step = tools.Application_Step
Service.step = tools.Service_Step
Service.provision = tools.Service_Provision
#networkflow
NetworkFlow.step = tools.NetworkFlow_Step
#server
EdgeServer.step = tools.EdgeServer_Step
EdgeServer.has_capacity_to_host = tools.has_capacity_to_host
#simulator
Simulator.step = tools.Simulator_Step
#networkswitch
NetworkSwitch.addQueue = tools.addQueue

In [5]:
#1.导入测试
#newworkflow schedule algorithm
simulate = Simulator(
    resource_management_algorithm = My_Schedule,
    network_flow_scheduling_algorithm = flow_share,
    stopping_criterion = Stop_func,
    resource_management_algorithm_parameters= {"mode":1,"k1":0.2,"k2":0.3,"alpha":0.5,"beta":0.5}
)

simulate.initialize(input_file="./test1_data.json")
datasets = Collect_Components()
printComponent(datasets,"User")
printComponent(datasets,"Application")
printComponent(datasets,"Service")
printComponent(datasets,"EdgeServer")
printComponent(datasets,"BaseStation")
printComponent(datasets,"NetworkSwitch")

User:
{'attributes': {'id': 1, 'coordinates': [0, 3], 'coordinates_trace': [], 'delays': {}, 'delay_slas': {'1': 45}, 'communication_paths': {}, 'making_requests': {}, 'mobility_model_parameters': {}}, 'relationships': {'access_patterns': {'1': {'class': 'CircularDurationAndIntervalAccessPattern', 'id': 1}}, 'mobility_model': None, 'applications': [{'class': 'Application', 'id': 1}], 'base_station': {'class': 'BaseStation', 'id': 1}}}

{'attributes': {'id': 2, 'coordinates': [0, 3], 'coordinates_trace': [], 'delays': {}, 'delay_slas': {'2': 30}, 'communication_paths': {}, 'making_requests': {}, 'mobility_model_parameters': {}}, 'relationships': {'access_patterns': {'2': {'class': 'CircularDurationAndIntervalAccessPattern', 'id': 2}}, 'mobility_model': None, 'applications': [{'class': 'Application', 'id': 2}], 'base_station': {'class': 'BaseStation', 'id': 1}}}

{'attributes': {'id': 3, 'coordinates': [0, 0], 'coordinates_trace': [], 'delays': {}, 'delay_slas': {'3': 30}, 'communication

In [29]:
#1.调度器步进
simulate.resource_management_algorithm_parameters["current_services"] = simulate.current_services
simulate.resource_management_algorithm(parameters=simulate.resource_management_algorithm_parameters)
print(simulate.current_services)
print(simulate.schedule.steps)
simulate.current_services = []

[Service_1, Service_2, Service_3]
2


In [30]:
#2.网络流步进
for agent in NetworkFlow.all():
    agent.step()

In [31]:

for server in EdgeServer.all():
    print(server.waiting_queue)
#未成功实现调度

[]
[]


In [ ]:
#3.服务器步进
for agent in EdgeServer.all():
    agent.step()

In [ ]:
# 服务步进
#3.服务器步进
for agent in Service.all():
    agent.step()

In [34]:
#4.应用步进
for agent in Application.all():
    agent.step()

In [35]:
#4.用户步进
for agent in User.all():
    agent.step()

In [36]:
print(User.all()[0].applications[0].status)
print(simulate.current_services)

wait
[]


In [37]:
#5.拓扑步进
for agent in Topology.all():
    agent.step()

In [38]:
simulate.schedule.steps+=1

In [28]:
#6. 判断仿真是否结束
print(simulate.stopping_criterion())

False


In [ ]:
#网络流步进


In [ ]:
#交换机步�?

In [ ]:

#链路步进